# Analyzing the Impact of Conference Awards on Researchers' Careers

This Jupyter notebook explores whether winning awards at top computer science conferences boosts researchers' future citations and publications. We compare award-winning authors (the "treated" group) with similar non-award authors (the "control" group) from the same conferences and years. Here's a simple step-by-step breakdown of what we did:

## Step 1: Load and Prepare Data
- We started by loading datasets: matched pairs of award winners and controls, lists of junior award authors, and unique conference-year combinations.
- We filtered out conferences without reliable data sources (like SIGCOMM and SIGMETRICS) to focus on 28 major ones (e.g., ICSE, CHI, AAAI).
- We identified source IDs (unique identifiers) for each conference's proceedings in the OpenAlex database.

## Step 2: Find Source IDs for Award Papers
- For each conference and year, we fetched the OpenAlex source ID by checking award-winning papers' metadata.
- This helped us pinpoint the exact proceedings volumes where papers were published.

## Step 3: Collect Non-Award Papers
- To create a control group, we randomly sampled 3 non-award papers per conference-year pair from the same proceedings.
- We excluded any papers by award winners to avoid overlap.
- This gave us a dataset of non-award authors and their papers.

## Step 4: Gather Author Profiles
- We fetched detailed profiles for all non-award authors from OpenAlex, including their yearly publication and citation counts.
- This data covers up to 11 years around each award year (5 years before and after).

## Step 5: Build Career Trajectories
- For each author, we calculated average citations and publications per year over a ±5 year window relative to the award year.
- We organized this into trajectories showing how their output changed over time.

## Step 6: Calculate "Lift" Metrics
- We computed the ratio of post-award metrics (years +1 to +5) to pre-award metrics (years -5 to -1) for citations and publications.
- This "lift" shows if awards lead to growth (ratios >1) or decline (ratios <1).
- We did this for both non-award authors and loaded pre-computed data for award authors.

## Step 7: Merge and Compare Groups
- We combined the non-award lift data with the award lift data into one dataset.
- We ran statistical tests (like Mann-Whitney U) to check if award winners have significantly higher lifts than non-winners.
- Results showed award authors generally have higher citation and publication lifts, with very low p-values indicating strong differences.

## Step 8: Sanity Checks and Summaries
- We verified the data quality, checked for missing values, and summarized key stats (e.g., medians, group sizes).
- The analysis suggests awards do correlate with career boosts, but more research could explore causality.

This notebook uses real data from OpenAlex (an academic database) and focuses on conferences like ICSE, CHI, and AAAI from 2000-2018. All steps include error handling and rate limiting to respect API rules. If you're new to this, think of it as comparing "winners" vs. "near-winners" in a race to see if the trophy changes their running speed! 🚀

In [1]:
# ── CELL 1: Imports & Config ─────────────────────────────────
import pandas as pd
import numpy as np
import requests, time, pickle
from pathlib import Path

ROOT    = Path("..")
MATCHED = ROOT / "data" / "matched"
FIG     = ROOT / "data" / "figures"
EMAIL   = "your@email.com"   # ← replace with your actual email

HEADERS = {"User-Agent": f"thesis-rq2/1.0 (mailto:{EMAIL})"}
BASE    = "https://api.openalex.org"

# Load award data
pairs   = pd.read_csv(MATCHED / "matched_pairs_clean.csv")
juniors = pd.read_csv(MATCHED / "junior_authors_all_conferences.csv")

# All award work IDs to exclude from non-award fetch
award_work_ids = set(juniors["work_id"].dropna().astype(str).unique())

# Unique (conf, year) pairs
cy_pairs = pairs.groupby(["conference","award_year"]).size().reset_index()[
    ["conference","award_year"]
].sort_values(["conference","award_year"]).reset_index(drop=True)

print(f"(conf, year) pairs:   {len(cy_pairs)}")
print(f"Award work IDs:       {len(award_work_ids)}")
print(f"Award authors:        {juniors['author_id'].nunique()}")
print(cy_pairs.head(5))


(conf, year) pairs:   268
Award work IDs:       470
Award authors:        603
  conference  award_year
0       AAAI        2000
1       AAAI        2004
2       AAAI        2007
3       AAAI        2008
4       AAAI        2010


In [2]:
# ── CELL 2f: Finalize source_ids, drop unresolvable conferences ──

# Manually add the confirmed IDs from Cell 2
source_ids = {
    "ICSE":       "S4306419842",
    "FSE":        "S4363608883",
    "CHI":        "S4363607743",
    "ACL":        "S4306420508",
    "UIST":       "S4306421131",
    "WWW":        "S4363608152",
    "INFOCOM":    "S4363607980",
    "SIGMOD":     "S4393915683",
    "KDD":        "S4306420424",
    "VLDB":       "S4306421142",
    "AAAI":       "S4210191458",
    "CIKM":       "S4306418063",
    "SOSP":       "S4306420989",
    "CVPR":       "S4363607701",
    "PODS":       "S4306420993",
    "OSDI":       "S4306420647",
    "ICML":       "S4306419644",
    "SIGIR":      "S4363608773",
    "MOBICOM":    "S4363608994",
    "PLDI":       "S4306420747",
    "S&P":        "S4363606603",
    "FOCS":       "S4306418447",
    "STOC":       "S4306421003",
    "IJCAI":      "S4306419999",
    "SODA":       "S4363608732",
    "ICCV":       "S4363607764",
    "NSDI":       "S4306420602",
    "NeurIPS":    "S4306420609",
    # SIGCOMM and SIGMETRICS dropped — no source metadata in OpenAlex
}

# Filter cy_pairs to only resolvable conferences
cy_pairs_filtered = cy_pairs[cy_pairs["conference"].isin(source_ids.keys())].reset_index(drop=True)

dropped = cy_pairs[~cy_pairs["conference"].isin(source_ids.keys())]
print(f"Conferences with source IDs: {len(source_ids)}/30")
print(f"Dropped (conf,year) pairs:   {len(dropped)}")
print(f"  {dropped['conference'].unique()}")
print(f"Remaining (conf,year) pairs: {len(cy_pairs_filtered)}")


Conferences with source IDs: 28/30
Dropped (conf,year) pairs:   14
  ['SIGCOMM' 'SIGMETRICS']
Remaining (conf,year) pairs: 254


In [3]:
# ── CELL 3 (Fixed): Dynamic source ID lookup per (conf, year) ──

def find_source_for_year(conf, year, venue_search_str, retries=3):
    """Find the best matching OpenAlex source ID for a conf in a specific year."""
    url = f"{BASE}/sources"
    # Search with year hint to find the right proceedings volume
    params = {
        "search":   f"{venue_search_str} {year}",
        "per-page": "5",
        "select":   "id,display_name,type,works_count",
    }
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, headers=HEADERS, timeout=20)
            results = r.json().get("results", [])
            if results:
                return results[0]["id"].split("/")[-1], results[0]["display_name"]
            # fallback: search without year
            params["search"] = venue_search_str
            r = requests.get(url, params=params, headers=HEADERS, timeout=20)
            results = r.json().get("results", [])
            if results:
                return results[0]["id"].split("/")[-1], results[0]["display_name"]
        except Exception:
            time.sleep(2 ** attempt)
    return None, None

# Shorter venue search strings — broader = better recall
VENUE_SEARCH = {
    "ICSE":    "International Conference on Software Engineering",
    "FSE":     "Foundations of Software Engineering",
    "CHI":     "Human Factors in Computing Systems",
    "ACL":     "Association for Computational Linguistics",
    "UIST":    "User Interface Software and Technology",
    "WWW":     "World Wide Web Conference",
    "INFOCOM": "IEEE INFOCOM",
    "SIGMOD":  "ACM SIGMOD",
    "KDD":     "Knowledge Discovery and Data Mining",
    "VLDB":    "Very Large Data Bases",
    "AAAI":    "AAAI Conference on Artificial Intelligence",
    "CIKM":    "Information and Knowledge Management",
    "SOSP":    "Symposium on Operating Systems Principles",
    "CVPR":    "Computer Vision and Pattern Recognition",
    "PODS":    "Principles of Database Systems",
    "OSDI":    "Operating Systems Design and Implementation",
    "ICML":    "International Conference on Machine Learning",
    "SIGIR":   "ACM SIGIR",
    "MOBICOM": "Mobile Computing and Networking",
    "PLDI":    "Programming Language Design and Implementation",
    "S&P":     "IEEE Symposium on Security and Privacy",
    "FOCS":    "Foundations of Computer Science",
    "STOC":    "Symposium on Theory of Computing",
    "IJCAI":   "International Joint Conference on Artificial Intelligence",
    "SODA":    "ACM-SIAM Symposium on Discrete Algorithms",
    "ICCV":    "International Conference on Computer Vision",
    "NSDI":    "Networked Systems Design and Implementation",
    "NeurIPS": "Neural Information Processing Systems",
}

# Test on a sample of known-empty pairs to verify fix
test_pairs = [
    ("AAAI",  2000), ("CHI",  2007), ("CVPR", 2006),
    ("FSE",   2016), ("S&P",  2009), ("SODA", 2010),
]

for conf, year in test_pairs:
    sid, name = find_source_for_year(conf, year, VENUE_SEARCH[conf])
    print(f"  {conf} {year}  →  {sid}  |  {str(name)[:60]}")
    time.sleep(0.5)


  AAAI 2000  →  S4210191458  |  Proceedings of the AAAI Conference on Artificial Intelligenc
  CHI 2007  →  S4363607743  |  CHI Conference on Human Factors in Computing Systems
  CVPR 2006  →  S4363607701  |  2022 IEEE/CVF Conference on Computer Vision and Pattern Reco
  FSE 2016  →  S4306418451  |  Foundations of Software Engineering
  S&P 2009  →  S4210233669  |  Proceedings - IEEE Symposium on Security and Privacy/Proceed
  SODA 2010  →  S4363608732  |  Proceedings of the Twentieth Annual ACM-SIAM Symposium on Di


In [4]:
# ── CELL 3d: Get source IDs directly from award paper metadata ──

# Grab one award work_id per (conf, year) from juniors
sample = (juniors
    .dropna(subset=["work_id"])
    .groupby(["conference","award_year"])["work_id"]
    .first()
    .reset_index()
)

award_source_map = {}  # (conf, year) → source_id

print("Fetching source IDs from award papers...\n")
for _, row in sample.iterrows():
    conf, year, wid = row["conference"], row["award_year"], row["work_id"]
    if conf not in source_ids:  # skip SIGCOMM/SIGMETRICS
        continue
    r = requests.get(f"{BASE}/works/{wid}", params={
        "select": "id,title,primary_location,locations"
    }, headers=HEADERS, timeout=20)
    if r.status_code != 200:
        continue
    data = r.json()

    # Try primary_location first, then all locations
    src = None
    pl = (data.get("primary_location") or {}).get("source")
    if pl and pl.get("id"):
        src = pl["id"].split("/")[-1]
    else:
        for loc in data.get("locations", []):
            s = (loc.get("source") or {})
            if s.get("id"):
                src = s["id"].split("/")[-1]
                break

    award_source_map[(conf, year)] = src
    status = f"✅ {src}" if src else "❌ none"
    print(f"  {conf:12s} {year}  →  {status}")
    time.sleep(0.3)

found    = sum(1 for v in award_source_map.values() if v)
missing  = [(k,v) for k,v in award_source_map.items() if not v]
print(f"\nSource IDs found: {found}/{len(award_source_map)}")
print(f"Still missing:    {[k for k,v in award_source_map.items() if not v][:10]}")


Fetching source IDs from award papers...

  AAAI         2000  →  ✅ S4306420577
  AAAI         2004  →  ✅ S4210169993
  AAAI         2007  →  ✅ S4306420577
  AAAI         2008  →  ✅ S4306420577
  AAAI         2010  →  ✅ S4210191458
  AAAI         2011  →  ✅ S4210191458
  AAAI         2012  →  ✅ S4210191458
  AAAI         2013  →  ✅ S4210191458
  AAAI         2016  →  ✅ S7407053046
  AAAI         2018  →  ✅ S4210191458
  ACL          2001  →  ❌ none
  ACL          2002  →  ❌ none
  ACL          2003  →  ❌ none
  ACL          2004  →  ✅ S4306400129
  ACL          2006  →  ❌ none
  ACL          2007  →  ✅ S4306420508
  ACL          2009  →  ❌ none
  ACL          2011  →  ✅ S4306420508
  ACL          2013  →  ✅ S4306420508
  ACL          2015  →  ❌ none
  ACL          2016  →  ❌ none
  ACL          2017  →  ✅ S4306400194
  ACL          2018  →  ✅ S4306400194
  CHI          2005  →  ❌ none
  CHI          2006  →  ❌ none
  CHI          2007  →  ❌ none
  CHI          2008  →  ✅ S1535467
  CHI

In [5]:
# ── CELL 3f: Retry missing pairs using all available work_ids ──

# For each missing (conf, year), try every work_id we have
missing_pairs = [(c, y) for (c, y), v in award_source_map.items() if not v]

print(f"Retrying {len(missing_pairs)} missing pairs with all work_ids...\n")

for conf, year in missing_pairs:
    if conf not in VENUE_SEARCH:
        continue
    # Get ALL work_ids for this (conf, year)
    work_ids = juniors[
        (juniors["conference"] == conf) & 
        (juniors["award_year"] == year)
    ]["work_id"].dropna().unique()

    found = False
    for wid in work_ids:
        r = requests.get(f"{BASE}/works/{wid}", params={
            "select": "id,primary_location,locations"
        }, headers=HEADERS, timeout=20)
        if r.status_code != 200:
            continue
        data = r.json()

        # Check primary_location
        src = None
        pl = (data.get("primary_location") or {}).get("source")
        if pl and pl.get("id"):
            src = pl["id"].split("/")[-1]
        # Check all locations
        if not src:
            for loc in data.get("locations", []):
                s = (loc.get("source") or {})
                if s.get("id"):
                    src = s["id"].split("/")[-1]
                    break

        if src:
            award_source_map[(conf, year)] = src
            print(f"  ✅ {conf:12s} {year}  →  {src}  (from wid {wid})")
            found = True
            break
        time.sleep(0.2)

    if not found:
        print(f"  ❌ {conf:12s} {year}  →  still none")
    time.sleep(0.3)

found_now = sum(1 for v in award_source_map.values() if v)
print(f"\nTotal covered after retry: {found_now}/{len(award_source_map)}")


Retrying 144 missing pairs with all work_ids...

  ❌ ACL          2001  →  still none
  ❌ ACL          2002  →  still none
  ❌ ACL          2003  →  still none
  ❌ ACL          2006  →  still none
  ❌ ACL          2009  →  still none
  ❌ ACL          2015  →  still none
  ❌ ACL          2016  →  still none
  ❌ CHI          2005  →  still none
  ❌ CHI          2006  →  still none
  ✅ CHI          2007  →  S53579087  (from wid https://openalex.org/W2152312062)
  ❌ CHI          2010  →  still none
  ✅ CHI          2012  →  S4306400895  (from wid https://openalex.org/W2166132393)
  ✅ CHI          2014  →  S4306401731  (from wid https://openalex.org/W2005511465)
  ✅ CHI          2015  →  S4306400230  (from wid https://openalex.org/W2077940530)
  ✅ CHI          2017  →  S4306400194  (from wid https://openalex.org/W4300093098)
  ✅ CHI          2018  →  S4306400766  (from wid https://openalex.org/W2795736453)
  ❌ CIKM         2004  →  still none
  ❌ CIKM         2005  →  still none
  ❌ CIKM   

In [6]:
# ── CELL 3g: Finalize covered pairs, save map ────────────────

import pickle

# Save the source map for reproducibility
with open(MATCHED / "award_source_map.pkl", "wb") as f:
    pickle.dump(award_source_map, f)

# Filter cy_pairs to only covered ones
cy_pairs_covered = cy_pairs_filtered[
    cy_pairs_filtered.apply(
        lambda r: bool(award_source_map.get((r["conference"], r["award_year"]))), axis=1
    )
].reset_index(drop=True)

print(f"Final (conf, year) pairs for RQ2: {len(cy_pairs_covered)}")
print(f"\nPer-conference coverage:")
for conf in sorted(cy_pairs_covered["conference"].unique()):
    total  = len(cy_pairs_filtered[cy_pairs_filtered["conference"] == conf])
    cov    = len(cy_pairs_covered[cy_pairs_covered["conference"] == conf])
    print(f"  {conf:12s}  {cov}/{total}")


Final (conf, year) pairs for RQ2: 134

Per-conference coverage:
  AAAI          10/10
  ACL           6/13
  CHI           11/14
  CIKM          2/10
  CVPR          4/9
  FOCS          5/7
  FSE           6/15
  ICCV          3/5
  ICML          5/8
  ICSE          13/16
  IJCAI         6/6
  INFOCOM       2/11
  KDD           3/11
  MOBICOM       4/7
  NSDI          2/4
  NeurIPS       4/4
  OSDI          6/8
  PLDI          2/7
  PODS          2/9
  S&P           5/7
  SIGIR         2/8
  SIGMOD        1/11
  SODA          3/5
  SOSP          7/9
  STOC          3/7
  UIST          5/12
  VLDB          9/10
  WWW           3/11


In [7]:
# ── CELL 4 (FIXED): Fetch non-award papers — random sample, not top-cited ──
import random

random.seed(42)  # reproducibility

def fetch_nonaward_papers(source_id, year, exclude_ids, n=3, retries=3):
    url = f"{BASE}/works"
    params = {
        "filter":   f"primary_location.source.id:{source_id},publication_year:{year},type:article",
        "select":   "id,title,authorships,cited_by_count",
        "per-page": "100",           # ← fetch a bigger pool
        "sort":     "display_name"   # ← alphabetical = no citation bias
    }
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, headers=HEADERS, timeout=20)
            r.raise_for_status()
            results = r.json().get("results", [])
            filtered = [w for w in results
                        if w["id"].split("/")[-1] not in exclude_ids]
            # ← randomly sample instead of taking first n
            return random.sample(filtered, min(n, len(filtered)))
        except Exception:
            time.sleep(2 ** attempt)
    return []


rows = []
empty_pairs = []

for i, row in cy_pairs_covered.iterrows():
    conf   = row["conference"]
    year   = row["award_year"]
    src_id = award_source_map[(conf, year)]
    papers = fetch_nonaward_papers(src_id, year, award_work_ids, n=3)

    if not papers:
        empty_pairs.append((conf, year))

    for p in papers:
        for a in p.get("authorships", []):
            if a.get("author") and a["author"].get("id"):
                rows.append({
                    "conference":      conf,
                    "award_year":      year,
                    "work_id":         p["id"].split("/")[-1],
                    "work_title":      p.get("title", ""),
                    "author_id":       a["author"]["id"].split("/")[-1],
                    "author_name":     a["author"].get("display_name", ""),
                    "author_position": a.get("author_position", ""),
                    "cited_by_count":  p.get("cited_by_count", 0),
                })

    if i % 20 == 0:
        print(f"  {i}/{len(cy_pairs_covered)}  —  {conf} {year}  —  {len(papers)} papers")
    time.sleep(1)

nonaward_raw = pd.DataFrame(rows)
nonaward_raw.to_csv(MATCHED / "nonaward_papers_raw.csv", index=False)

print(f"\nDone.")
print(f"Total author rows:   {len(nonaward_raw)}")
print(f"Unique papers:       {nonaward_raw['work_id'].nunique()}")
print(f"Unique authors:      {nonaward_raw['author_id'].nunique()}")
print(f"Empty (conf,year):   {len(empty_pairs)}  →  {empty_pairs[:5]}")
print(nonaward_raw.head(3))


  0/134  —  AAAI 2000  —  3 papers
  20/134  —  CHI 2012  —  3 papers
  40/134  —  FSE 2014  —  3 papers
  60/134  —  ICSE 2013  —  3 papers
  80/134  —  NSDI 2008  —  3 papers
  100/134  —  S&P 2017  —  3 papers
  120/134  —  UIST 2017  —  3 papers

Done.
Total author rows:   1373
Unique papers:       393
Unique authors:      1352
Empty (conf,year):   0  →  []
  conference  award_year      work_id  \
0       AAAI        2000  W1573508696   
1       AAAI        2000  W1573508696   
2       AAAI        2000  W1573508696   

                                          work_title    author_id  \
0  Selective Sampling with Co-Testing: Preliminar...  A5011344987   
1  Selective Sampling with Co-Testing: Preliminar...  A5045363082   
2  Selective Sampling with Co-Testing: Preliminar...  A5089542402   

         author_name author_position  cited_by_count  
0         Ion Muslea           first               3  
1      Steven Minton          middle               3  
2  Craig A. Knoblock         

In [8]:
# ── CELL 5: Fetch OpenAlex profiles for non-award authors ────

def fetch_author_profile(author_id, retries=3):
    url = f"{BASE}/authors/{author_id}"
    params = {"select": "id,display_name,counts_by_year,works_count,cited_by_count"}
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, headers=HEADERS, timeout=20)
            if r.status_code == 404:
                return None
            r.raise_for_status()
            return r.json()
        except Exception:
            time.sleep(2 ** attempt)
    return None

author_ids = nonaward_raw["author_id"].unique()
profiles   = {}
missing    = []

for i, aid in enumerate(author_ids):
    prof = fetch_author_profile(aid)
    if prof:
        profiles[aid] = prof
    else:
        missing.append(aid)
    if i % 200 == 0:
        print(f"  {i}/{len(author_ids)}  —  fetched: {len(profiles)}  missing: {len(missing)}")
    time.sleep(0.5)

# Save checkpoint
import pickle
with open(MATCHED / "nonaward_profiles.pkl", "wb") as f:
    pickle.dump(profiles, f)

print(f"\nDone.")
print(f"Profiles fetched:  {len(profiles)}/{len(author_ids)}")
print(f"Missing/failed:    {len(missing)}")


  0/1352  —  fetched: 1  missing: 0
  200/1352  —  fetched: 201  missing: 0
  400/1352  —  fetched: 401  missing: 0
  600/1352  —  fetched: 601  missing: 0
  800/1352  —  fetched: 801  missing: 0
  1000/1352  —  fetched: 1001  missing: 0
  1200/1352  —  fetched: 1201  missing: 0

Done.
Profiles fetched:  1352/1352
Missing/failed:    0


In [9]:
# ── CELL 6: Build ±5 year trajectories for non-award authors ──

def build_trajectory(author_id, award_year, profile, window=5):
    cby = {e["year"]: e for e in profile.get("counts_by_year", [])}
    records = []
    for rel in range(-window, window + 1):
        yr    = award_year + rel
        entry = cby.get(yr, {})
        records.append({
            "author_id":   author_id,
            "award_year":  award_year,
            "rel_year":    rel,
            "pubs":        entry.get("works_count", 0),
            "cits":        entry.get("cited_by_count", 0),
        })
    return records

traj_rows = []
skipped   = 0

for _, row in nonaward_raw.iterrows():
    aid = row["author_id"]
    if aid not in profiles:
        skipped += 1
        continue
    traj_rows.extend(
        build_trajectory(aid, row["award_year"], profiles[aid])
    )

nonaward_traj = (
    pd.DataFrame(traj_rows)
    .drop_duplicates(subset=["author_id", "award_year", "rel_year"])
    .reset_index(drop=True)
)

nonaward_traj.to_csv(MATCHED / "nonaward_trajectories.csv", index=False)

print(f"Trajectory rows:      {len(nonaward_traj)}")
print(f"Unique authors:       {nonaward_traj['author_id'].nunique()}")
print(f"Skipped (no profile): {skipped}")
print(f"\nSample (one author):")
print(nonaward_traj[nonaward_traj["author_id"] == nonaward_traj["author_id"].iloc[0]])


Trajectory rows:      15026
Unique authors:       1352
Skipped (no profile): 0

Sample (one author):
      author_id  award_year  rel_year  pubs  cits
0   A5011344987        2000        -5     0     0
1   A5011344987        2000        -4     0     0
2   A5011344987        2000        -3     1    25
3   A5011344987        2000        -2     8   352
4   A5011344987        2000        -1     6   753
5   A5011344987        2000         0     3   247
6   A5011344987        2000         1     4   573
7   A5011344987        2000         2     5   354
8   A5011344987        2000         3     3   201
9   A5011344987        2000         4     1    60
10  A5011344987        2000         5     2    44


In [10]:
# ── CELL 7 (FIXED): Compute pre/post lift as RATIO ──────────────────────────

def compute_lift_ratio(traj_df, pre=(-5,-1), post=(1,5)):
    rows = []
    for (aid, yr), grp in traj_df.groupby(["author_id","award_year"]):
        pre_df  = grp[grp["rel_year"].between(pre[0],  pre[1])]
        post_df = grp[grp["rel_year"].between(post[0], post[1])]

        pre_cits  = pre_df["cits"].mean()
        post_cits = post_df["cits"].mean()
        pre_pubs  = pre_df["pubs"].mean()
        post_pubs = post_df["pubs"].mean()

        rows.append({
            "author_id":        aid,
            "award_year":       yr,
            "pre_avg_citations":  pre_cits,
            "post_avg_citations": post_cits,
            "pre_avg_works":      pre_pubs,
            "post_avg_works":     post_pubs,
            # ← ratio, not difference — matches author_lift.csv
            "lift_citations": post_cits / pre_cits if pre_cits > 0 else None,
            "lift_works":     post_pubs / pre_pubs if pre_pubs > 0 else None,
        })
    return pd.DataFrame(rows)


nonaward_lift = compute_lift_ratio(nonaward_traj)

# Merge metadata
nonaward_lift = nonaward_lift.merge(
    nonaward_raw[["author_id","award_year","conference","author_position"]].drop_duplicates(),
    on=["author_id","award_year"], how="left"
)

nonaward_lift["group"] = "nonaward"
nonaward_lift.to_csv(MATCHED / "nonaward_lift.csv", index=False)

print(f"Non-award lift rows: {len(nonaward_lift)}")
print(f"\nOverall lift summary:")
print(nonaward_lift[["lift_citations","lift_works"]].describe().round(2))
print(f"\nBy position:")
print(nonaward_lift.groupby("author_position")[["lift_citations","lift_works"]].median().round(2))


Non-award lift rows: 1372

Overall lift summary:
       lift_citations  lift_works
count         1116.00     1245.00
mean            10.69        2.44
std             63.31        7.13
min              0.00        0.00
25%              0.40        0.72
50%              1.00        1.29
75%              3.05        2.43
max           1039.00      211.00

By position:
                 lift_citations  lift_works
author_position                            
first                      0.93        1.27
last                       0.81        1.22
middle                     1.21        1.33


In [11]:
# ── CELL 8: Load award lift and merge with non-award lift ─────

award_lift = pd.read_csv(MATCHED / "author_lift.csv")

print("Award lift columns:", award_lift.columns.tolist())
print("Award lift shape:  ", award_lift.shape)
print(award_lift.head(3))


Award lift columns: ['author_id', 'author_name', 'treatment', 'conference', 'award_year', 'pre_avg_citations', 'post_avg_citations', 'pre_avg_works', 'post_avg_works', 'lift_citations', 'lift_works']
Award lift shape:   (961, 11)
                          author_id       author_name  treatment conference  \
0  https://openalex.org/A5014823249      Jincheng Mei          1       AAAI   
1  https://openalex.org/A5069646529  Adhiguna Kuncoro          1        ACL   
2  https://openalex.org/A5032195981      Sang-Gyun An          1        CHI   

   award_year  pre_avg_citations  post_avg_citations  pre_avg_works  \
0        2018              19.75                48.0            2.0   
1        2018             319.00               145.6            4.0   
2        2018              27.00                15.0            2.0   

   post_avg_works  lift_citations  lift_works  
0        4.333333        2.430380    2.166667  
1        4.600000        0.456426    1.150000  
2        2.000000       

In [12]:
# ── CELL 8b: Verify lift calculation type in award_lift ───────

# Check if lift_citations is ratio or difference
sample = award_lift[["pre_avg_citations","post_avg_citations","lift_citations"]].head(10)
sample["ratio_check"] = sample["post_avg_citations"] / sample["pre_avg_citations"].replace(0, float("nan"))
sample["diff_check"]  = sample["post_avg_citations"] - sample["pre_avg_citations"]
print(sample.to_string())


   pre_avg_citations  post_avg_citations  lift_citations  ratio_check  diff_check
0          19.750000                48.0        2.430380     2.430380   28.250000
1         319.000000               145.6        0.456426     0.456426 -173.400000
2          27.000000                15.0        0.555556     0.555556  -12.000000
3           2.500000                17.5        7.000000     7.000000   15.000000
4         141.500000                63.4        0.448057     0.448057  -78.100000
5           8.000000                 NaN             NaN          NaN         NaN
6          64.000000               231.2        3.612500     3.612500  167.200000
7         447.000000                 9.0        0.020134     0.020134 -438.000000
8          78.500000                12.0        0.152866     0.152866  -66.500000
9          45.666667                60.6        1.327007     1.327007   14.933333


In [13]:
# ── CELL 9 (SIMPLIFIED): Merge award + nonaward lift ──────────────────────

# Load the already-computed ratio lift from Cell 7
nonaward_lift_ratio = pd.read_csv(MATCHED / "nonaward_lift.csv")
nonaward_lift_ratio["group"] = "nonaward"

# Align award_lift columns
award_lift_aligned = award_lift[[
    "author_id", "award_year", "conference",
    "pre_avg_citations", "post_avg_citations",
    "pre_avg_works", "post_avg_works",
    "lift_citations", "lift_works",
    "treatment"
]].copy()
award_lift_aligned["group"] = "award"

# Stack both
combined = pd.concat([
    nonaward_lift_ratio,
    award_lift_aligned
], ignore_index=True)

combined.to_csv(MATCHED / "rq2_combined_lift.csv", index=False)

print(f"Combined rows: {len(combined)}")
print(f"\nGroup counts:")
print(combined["group"].value_counts())
print(f"\nMedian lift by group:")
print(combined.groupby("group")[["lift_citations","lift_works"]].median().round(3))


Combined rows: 2333

Group counts:
group
nonaward    1372
award        961
Name: count, dtype: int64

Median lift by group:
          lift_citations  lift_works
group                               
award              1.773       1.696
nonaward           1.000       1.286


In [14]:
# ── CELL 10: Sanity check stats ───────────────────────────────────────────
from scipy import stats

# Quick overall check
for metric in ["lift_citations", "lift_works"]:
    a = combined[combined["group"] == "award"][metric].dropna()
    b = combined[combined["group"] == "nonaward"][metric].dropna()
    stat, p = stats.mannwhitneyu(a, b, alternative="greater")
    print(f"{metric}: award median={a.median():.3f}  nonaward median={b.median():.3f}  "
          f"U={stat:.0f}  p={p:.4f}")

print()
print(f"Combined rows:  {len(combined)}")
print(f"Group counts:\n{combined['group'].value_counts()}")
print(f"\nMedian lift by group:")
print(combined.groupby("group")[["lift_citations","lift_works"]].median().round(3))


lift_citations: award median=1.773  nonaward median=1.000  U=618672  p=0.0000
lift_works: award median=1.696  nonaward median=1.286  U=695650  p=0.0000

Combined rows:  2333
Group counts:
group
nonaward    1372
award        961
Name: count, dtype: int64

Median lift by group:
          lift_citations  lift_works
group                               
award              1.773       1.696
nonaward           1.000       1.286
